In [1]:
import pandas as pd
import os
import numpy as np
import re

In [2]:
pd.set_option('display.max_columns', None)

# <span style="color:blue;">**Cancer**</span>

In [3]:
study = "Cancer"

# Data preparation

## A. DICOM

#### Visit [parameter exlanation][peid] for more information

[peid]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/Doc.aspx?sourcedoc=%7B7B05A758-0D43-40AF-9177-E9C7522D2649%7D&file=Notes_parameters%20explaination.docx&action=default&mobileredirect=true 


In [4]:
file_path = "../Data/dicom_tag.xlsx"
dicom = pd.read_excel(file_path)

In [5]:
dicom["PatientID"].unique().size

5515

In [6]:
PIDs = dicom["PatientID"].unique()

#### Create a list of unique DICOM entries to serve as the reference linking the EHR to the images available for specific patient visits (as images are limited to certain visits, not all).

#### See shared parameters in [parameter exlanation][peid] to link data

[peid]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/Doc.aspx?sourcedoc=%7B7B05A758-0D43-40AF-9177-E9C7522D2649%7D&file=Notes_parameters%20explaination.docx&action=default&mobileredirect=true 

**shared parameters:**

DICOM ↔︎ EHR (pathology)

PatientID ↔︎ PATIENT_STUDY_ID 

AccessionNumber ↔︎ ACCESSION_NUMBER 

PatientBirthDate ↔︎ BIRTH_DATE

In [7]:
dicom.rename(columns={'PatientID': 'PATIENT_STUDY_ID', 'AccessionNumber': 'ACCESSION_NUMBER', 'PatientBirthDate': 'BIRTH_DATE'}, inplace=True)

In [8]:
dicom

,PATIENT_STUDY_ID,ACCESSION_NUMBER,BIRTH_DATE,PatientAge,PatientSex,StudyDate,StudyTime,AcquisitionDate,Modality,Manufacturer,ManufacturerModelName,StudyDescription,SeriesNumber,SeriesDescription,Exposure,Rows,Columns,PixelSpacing,study,side,series
0,4330018595,60103700,1965-07-01,054Y,F,2020-06-02,09:10:52,NaN,MG,"R2 Technology, Inc.",Cenova,DIAG DIG MAMMO RIGHT ALL VIEWS WITH TOMOSYNTHESIS,1.0,Hologic R2 ImageChecker CAD SC,NaN,1500.0,1250.0,NaN,DIAG,NaN,NaN
1,4330018595,60103700,1965-07-01,NaN,F,2020-06-02,09:10:51,NaN,MG,"R2 Technology, Inc.",Cenova,DIAG DIG MAMMO RIGHT ALL VIEWS WITH TOMOSYNTHESIS,1.0,Hologic R2 ImageChecker CAD SC,NaN,1500.0,1250.0,NaN,DIAG,NaN,NaN
2,4330018595,60103700,1965-07-01,NaN,F,2020-06-02,09:10:51,2020-06-02,MG,"HOLOGIC, Inc.",Selenia Dimensions,DIAG DIG MAMMO RIGHT ALL VIEWS WITH TOMOSYNTHESIS,71100000.0,R ML,126.0,3328.0,2560.0,0.038889\0.038889,DIAG,R,NaN
3,4330018595,60103700,1965-07-01,NaN,F,2020-06-02,09:10:51,NaN,MG,"HOLOGIC, Inc.",Selenia Dimensions,DIAG DIG MAMMO RIGHT ALL VIEWS WITH TOMOSYNTHESIS,71300000.0,R XCCL Intelligent 2D,63.0,3328.0,2560.0,0.064318\0.064318,DIAG,R,IN2D
4,4330018595,60103700,1965-07-01,NaN,F,2020-06-02,09:10:51,NaN,MG,"HOLOGIC, Inc.",Selenia Dimensions,DIAG DIG MAMMO RIGHT ALL VIEWS WITH TOMOSYNTHESIS,71300000.0,R ML Intelligent 2D,57.0,3328.0,2560.0,0.064619\0.064619,DIAG,R,IN2D
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
136228,4339959661,62216536,1971-07-01,NaN,F,2019-10-28,02:45:37,2019-10-28,MG,"HOLOGIC, Inc.",Selenia Dimensions,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMOSYN,72100000.0,L CC Tomosynthesis Projection,72.0,425.0,266.0,NaN,SCREEN,L,DBT
136229,4339959661,62216536,1971-07-01,NaN,F,2019-10-28,02:45:37,2019-10-28,MG,"HOLOGIC, Inc.",Selenia Dimensions,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMOSYN,72100000.0,R CC Tomosynthesis Projection,74.0,425.0,266.0,NaN,SCREEN,R,DBT
136230,4339959661,62216536,1971-07-01,NaN,F,2019-10-28,02:45:37,2019-10-28,MG,"HOLOGIC, Inc.",Selenia Dimensions,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMOSYN,72100000.0,L MLO Tomosynthesis Projection,78.0,425.0,266.0,NaN,SCREEN,L,DBT
136231,4339959661,62216536,1971-07-01,NaN,F,2019-10-28,18:20:08,NaN,PR,Philips Medical Systems,iSite Enterprise,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMOSYN,1.0,5DBAD14E0,NaN,NaN,NaN,NaN,SCREEN,NaN,NaN


In [9]:
# Define the key identifier columns and columns to extract
key_columns = ['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'StudyDate', 'BIRTH_DATE', 'PatientSex']
extract_columns = ['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'StudyDate', 'StudyTime', 'study']
unique_dicom = dicom.drop_duplicates(subset=key_columns, keep='first')
unique_dicom = unique_dicom[extract_columns]
unique_dicom.reset_index(drop=True, inplace=True)

In [10]:
unique_dicom['image_available'] = 1

In [11]:
unique_dicom

,PATIENT_STUDY_ID,ACCESSION_NUMBER,StudyDate,StudyTime,study,image_available
0,4330018595,60103700,2020-06-02,09:10:52,DIAG,1
1,4330018595,60690108,2020-06-02,08:33:36,SCREEN,1
2,4330018595,63737104,2019-08-19,02:11:20,DIAG,1
3,4330029102,64888584,2019-05-16,08:32:09,SCREEN,1
4,4330044371,61647696,2020-02-06,15:37:19,SCREEN,1
...,...,...,...,...,...,...
15135,4339943307,63434181,2019-06-17,17:07:31,DIAG,1
15136,4339943307,63570279,2019-05-23,18:36:05,SCREEN,1
15137,4339945522,62906727,2019-10-18,03:00:54,DIAG,1
15138,4339945522,450264459,2022-10-14,11:54:03,NaN,1


#### Add "dbt_available" column, if available = 1

In [12]:
dicom_DBT = dicom[dicom["series"]=="DBT"]
key_columns = ['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'StudyDate', 'BIRTH_DATE', 'PatientSex']
extract_columns = ['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'StudyDate', 'study', 'series']
unique_dicom_DBT = dicom_DBT.drop_duplicates(subset=key_columns, keep='first')[extract_columns]

In [13]:
unique_dicom = unique_dicom.merge(
    unique_dicom_DBT, 
    on=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'StudyDate', 'study'],
    how='left')

In [14]:
unique_dicom = unique_dicom.rename(columns={'series': 'dbt_available'})
unique_dicom['dbt_available'] = unique_dicom['dbt_available'].replace('DBT', 1)

/var/folders/g9/k5r102q54f126nrlr692zy6c0000gn/T/ipykernel_35586/3012130192.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  unique_dicom['dbt_available'] = unique_dicom['dbt_available'].replace('DBT', 1)


In [15]:
unique_dicom

,PATIENT_STUDY_ID,ACCESSION_NUMBER,StudyDate,StudyTime,study,image_available,dbt_available
0,4330018595,60103700,2020-06-02,09:10:52,DIAG,1,NaN
1,4330018595,60690108,2020-06-02,08:33:36,SCREEN,1,NaN
2,4330018595,63737104,2019-08-19,02:11:20,DIAG,1,NaN
3,4330029102,64888584,2019-05-16,08:32:09,SCREEN,1,NaN
4,4330044371,61647696,2020-02-06,15:37:19,SCREEN,1,NaN
...,...,...,...,...,...,...,...
15135,4339943307,63434181,2019-06-17,17:07:31,DIAG,1,NaN
15136,4339943307,63570279,2019-05-23,18:36:05,SCREEN,1,NaN
15137,4339945522,62906727,2019-10-18,03:00:54,DIAG,1,NaN
15138,4339945522,450264459,2022-10-14,11:54:03,NaN,1,NaN


## B. Electric Health Record

#### Visit [parameter exlanation][peid] for more information

[peid]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/Doc.aspx?sourcedoc=%7B7B05A758-0D43-40AF-9177-E9C7522D2649%7D&file=Notes_parameters%20explaination.docx&action=default&mobileredirect=true 


In [16]:
file_path = "../Data/parameters of interest.xlsx"
file_name = pd.ExcelFile(file_path).sheet_names
file_name

['pathology',
 'pathology_findings',
 'patient_data_ie',
 'family_hx',
 'patient_demo',
 'vitals',
 'risk_factors',
 'enteredit_findings',
 'hormonal_mens']

---

# I. Extract <span style="color:blue;">**Cancer**</span> 
#### output file = "cancer_cohort", N=1956

#### See [inclusion criteria][ic] to for filtering

[ic]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/doc.aspx?sourcedoc=%7B9c403413-a771-4830-98ab-15171872c4dd%7D&action=edit

## 1. Merge with enteredit_findings 

* Columns: COMPOSITION_NAME, FINDING_LOCATION, FINDING_CATEGORY, FINDING_REC, EXAM_COMPLETED_DATE
* Cancer: 
    * FINDING_CATEGORY = 4, 4A, 4B, 4C, 5

In [17]:
fn = file_name[7]
fn

'enteredit_findings'

In [18]:
file_path = os.path.join("../Data/", study, "Cleaned", fn + ".xlsx")
BIRADS = pd.read_excel(file_path)

In [19]:
set_dicom_PIDs = set(PIDs)
set_BIRADS_PIDs = set(BIRADS["PATIENT_STUDY_ID"].unique())

common_PIDs = set_dicom_PIDs & set_BIRADS_PIDs

In [20]:
len(common_PIDs)

5509

In [21]:
unique_dicom[unique_dicom["PATIENT_STUDY_ID"]==4330344296]

,PATIENT_STUDY_ID,ACCESSION_NUMBER,StudyDate,StudyTime,study,image_available,dbt_available
66,4330344296,62861703,2019-12-20,02:44:43,SCREEN,1,NaN
67,4330344296,67469599,2020-12-31,08:39:44,SCREEN,1,NaN
68,4330344296,71644307,2017-07-31,02:23:29,SCREEN,1,NaN
69,4330344296,76610889,2018-10-31,16:35:19,SCREEN,1,NaN
70,4330344296,76695188,2018-11-08,15:23:00,DIAG,1,NaN


In [22]:
BIRADS[BIRADS["PATIENT_STUDY_ID"]==4330344296]

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count
402,4330344296,71644307,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2017-07-31,2
403,4330344296,76610889,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2018-10-31,1
404,4330344296,76610889,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2018-10-31,1
405,4330344296,76695188,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2018-11-08,1
406,4330344296,62861703,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2019-12-20,2
407,4330344296,67469599,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2020-12-31,1
408,4330344296,67469599,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2020-12-31,1
409,4330344296,67156697,Heterogeneously dense (51% - 75%),NaN,4A - Suspicious abnormality - biopsy should be...,B-Biopsy should be considered,2021-01-25,2
410,4330344296,453737129,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2022-02-10,1


In [23]:
merge = pd.merge(
    BIRADS, 
    unique_dicom,
    on=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER'], 
    how='left'
)

In [24]:
merge[merge["PATIENT_STUDY_ID"]==4330344296]

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count,StudyDate,StudyTime,study,image_available,dbt_available
404,4330344296,71644307,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2017-07-31,2,2017-07-31,02:23:29,SCREEN,1.0,NaN
405,4330344296,76610889,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2018-10-31,1,2018-10-31,16:35:19,SCREEN,1.0,NaN
406,4330344296,76610889,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2018-10-31,1,2018-10-31,16:35:19,SCREEN,1.0,NaN
407,4330344296,76695188,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2018-11-08,1,2018-11-08,15:23:00,DIAG,1.0,NaN
408,4330344296,62861703,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2019-12-20,2,2019-12-20,02:44:43,SCREEN,1.0,NaN
409,4330344296,67469599,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2020-12-31,1,2020-12-31,08:39:44,SCREEN,1.0,NaN
410,4330344296,67469599,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2020-12-31,1,2020-12-31,08:39:44,SCREEN,1.0,NaN
411,4330344296,67156697,Heterogeneously dense (51% - 75%),NaN,4A - Suspicious abnormality - biopsy should be...,B-Biopsy should be considered,2021-01-25,2,NaN,NaN,NaN,NaN,NaN
412,4330344296,453737129,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2022-02-10,1,NaN,NaN,NaN,NaN,NaN


In [25]:
merge_copy = merge[merge["PATIENT_STUDY_ID"].isin(PIDs)].copy()

In [26]:
merge_copy["PATIENT_STUDY_ID"].unique().size

5509

## 2. Merge with pathology

### preparation: sort by dates

* Columns: BX_ID, PATHOLOGY_DATE, LESION_CLASS, SIDE
* Cancer: 
    * LESION_CLASS = 'Malignant'

In [27]:
fn = file_name[0]
fn

'pathology'

In [28]:
file_path = os.path.join("../Data/", study, "Cleaned", fn + ".xlsx")
pathology = pd.read_excel(file_path)

In [29]:
pathology_sort = pathology.copy()
merge_sort = merge_copy.copy()

pathology_sort['PATHOLOGY_DATE'] = pd.to_datetime(pathology_sort['PATHOLOGY_DATE'])
merge_sort['EXAM_COMPLETED_DATE'] = pd.to_datetime(merge_sort['EXAM_COMPLETED_DATE'])

pathology_sort.sort_values(by =['PATIENT_STUDY_ID', 'PATHOLOGY_DATE'],inplace=True)
merge_sort.sort_values(by =['PATIENT_STUDY_ID', 'EXAM_COMPLETED_DATE'],inplace=True)

### Store <span style="color:blue;">**cancer**</span> record only

#### For each patient, find if LESION_CLASS = 'Malignant' ever exists in their records, and only store the record with 'Malignant' findings.**

In [30]:
cancer_cohort = None
PIDs_without_pathology = []
PIDs_without_malignant = []
PIDs_with_malignant = []

for pid in PIDs:
    
    # merge_asof can only deal with one patient ID at a time, and the date needs to be sorted ascending
    
    path = pathology_sort[pathology_sort["PATIENT_STUDY_ID"]==pid]
    merg = merge_sort[merge_sort["PATIENT_STUDY_ID"]==pid]
    
    if path.size == 0:
        
        PIDs_without_pathology.append(pid)
#         print("skip")
    
    else:
    
        closest_match = pd.merge_asof(
        left=path,
        right=merg,
        left_on='PATHOLOGY_DATE',          # The date we are matching FROM
        right_on='EXAM_COMPLETED_DATE',    # The date we are matching TO
        by='PATIENT_STUDY_ID',             # The grouping key (Int64)
        direction='backward',             # Finds the closest date (after)
        allow_exact_matches=True,
        tolerance=pd.Timedelta(days=365)   # maximum time interval to be considered
        )
        
        
        # Check if LESION_CLASS = 'Malignant' ever exsits for the patient, if none, skip
        column_to_check = 'LESION_CLASS' 
        closest_match[column_to_check] = closest_match[column_to_check].astype(str)
        lesion_terms = 'Malignant'
        lesion_mask = closest_match[column_to_check].str.contains(lesion_terms, case=False, na=False, regex=True)
        
        if closest_match.loc[lesion_mask].size==0: 
            PIDs_without_malignant.append(pid)
#             print("no malignant")
        else:
            PIDs_with_malignant.append(pid)
            combine = merg.merge(closest_match, how='outer')
            cancer_cohort = pd.concat([cancer_cohort, combine])

In [31]:
cancer_cohort.reset_index(inplace=True, drop=True)
print("‼️ # of patients without pathology:", len(PIDs_without_pathology))
print("‼️ # of patients with pathology:", len(PIDs_without_malignant)+len(PIDs_with_malignant))
print("‼️ # of patients without malignant:",len(PIDs_without_malignant))
print("‼️ # of patients with malignant:",len(PIDs_with_malignant))

‼️ # of patients without pathology: 305
‼️ # of patients with pathology: 5210
‼️ # of patients without malignant: 3254
‼️ # of patients with malignant: 1956


In [34]:
cancer_cohort_sort = cancer_cohort.sort_values(['PATIENT_STUDY_ID', 'EXAM_COMPLETED_DATE', 'ACCESSION_NUMBER']).reset_index(drop=True)

In [36]:
cancer_cohort_sort[cancer_cohort_sort["PATIENT_STUDY_ID"]==4330344296]

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count,StudyDate,StudyTime,study,image_available,dbt_available,BX_ID,PATHOLOGY_DATE,LESION_CLASS,SIDE
108,4330344296,71644307.0,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2017-07-31,2.0,2017-07-31,02:23:29,SCREEN,1.0,NaN,NaN,NaT,NaN,NaN
109,4330344296,76610889.0,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2018-10-31,1.0,2018-10-31,16:35:19,SCREEN,1.0,NaN,NaN,NaT,NaN,NaN
110,4330344296,76610889.0,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2018-10-31,1.0,2018-10-31,16:35:19,SCREEN,1.0,NaN,NaN,NaT,NaN,NaN
111,4330344296,76695188.0,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2018-11-08,1.0,2018-11-08,15:23:00,DIAG,1.0,NaN,NaN,NaT,NaN,NaN
112,4330344296,62861703.0,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2019-12-20,2.0,2019-12-20,02:44:43,SCREEN,1.0,NaN,NaN,NaT,NaN,NaN
113,4330344296,67469599.0,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2020-12-31,1.0,2020-12-31,08:39:44,SCREEN,1.0,NaN,NaN,NaT,NaN,NaN
114,4330344296,67469599.0,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2020-12-31,1.0,2020-12-31,08:39:44,SCREEN,1.0,NaN,NaN,NaT,NaN,NaN
115,4330344296,67156697.0,Heterogeneously dense (51% - 75%),NaN,4A - Suspicious abnormality - biopsy should be...,B-Biopsy should be considered,2021-01-25,2.0,NaN,NaN,NaN,NaN,NaN,470168.0,2021-02-10,Malignant,R
116,4330344296,67156697.0,Heterogeneously dense (51% - 75%),NaN,4A - Suspicious abnormality - biopsy should be...,B-Biopsy should be considered,2021-01-25,2.0,NaN,NaN,NaN,NaN,NaN,470167.0,2021-02-10,Benign,R
117,4330344296,67156697.0,Heterogeneously dense (51% - 75%),NaN,4A - Suspicious abnormality - biopsy should be...,B-Biopsy should be considered,2021-01-25,2.0,NaN,NaN,NaN,NaN,NaN,478775.0,2021-03-05,Malignant,R


In [37]:
output_file = os.path.join("../Data/", study, 'cancer_cohort' + ".xlsx")
cancer_cohort_sort.to_excel(output_file, index=False)

## 3. Fill missing <span style="color:blue"> **Study**</span> (SCREEN, DIAG) with patient_data_ie

#### <span style="color:#FF6347;">**READ**</span> file

In [42]:
file_path = "../Data/Cancer/cancer_cohort.xlsx"
cancer_cohort = pd.read_excel(file_path)

In [43]:
file_path = "../Data/Cancer/patient_data_ie_tag.xlsx"
pdi = pd.read_excel(file_path)

In [44]:
pdi.columns

Index(['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'ORG_NAME', 'EXAM_CODE',
       'EXAM_NAME', 'COMPLETED_DATE', 'ORDER_DATE', 'EXAM_TYPE',
       'FIRST_MAMMO_IND', 'LAST_ACTIVITY_DATE', 'INTERNAL_EXAM_ID',
       'PATIENT_ID', 'study'],
      dtype='object')

In [45]:
extract_cols = ['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'EXAM_TYPE', 'study']

In [47]:
cancer_cohort_copy = cancer_cohort.copy()
pdi_copy = pdi[extract_cols].copy()

In [48]:
cancer_cols = cancer_cohort_copy.columns
pdi_cols = pdi_copy.columns

In [49]:
merge = pd.merge(cancer_cohort_copy, pdi_copy, how="left", on=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER'])

In [50]:
merge[merge["PATIENT_STUDY_ID"]==4330344296]

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count,StudyDate,StudyTime,study_x,image_available,dbt_available,BX_ID,PATHOLOGY_DATE,LESION_CLASS,SIDE,EXAM_TYPE,study_y
108,4330344296,71644307.0,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2017-07-31,2.0,2017-07-31,02:23:29,SCREEN,1.0,NaN,NaN,NaT,NaN,NaN,Mammography,SCREEN
109,4330344296,76610889.0,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2018-10-31,1.0,2018-10-31,16:35:19,SCREEN,1.0,NaN,NaN,NaT,NaN,NaN,Mammography,SCREEN
110,4330344296,76610889.0,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2018-10-31,1.0,2018-10-31,16:35:19,SCREEN,1.0,NaN,NaN,NaT,NaN,NaN,Mammography,SCREEN
111,4330344296,76695188.0,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2018-11-08,1.0,2018-11-08,15:23:00,DIAG,1.0,NaN,NaN,NaT,NaN,NaN,Mammography,DIAG
112,4330344296,62861703.0,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2019-12-20,2.0,2019-12-20,02:44:43,SCREEN,1.0,NaN,NaN,NaT,NaN,NaN,Mammography,SCREEN
113,4330344296,67469599.0,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2020-12-31,1.0,2020-12-31,08:39:44,SCREEN,1.0,NaN,NaN,NaT,NaN,NaN,Mammography,SCREEN
114,4330344296,67469599.0,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2020-12-31,1.0,2020-12-31,08:39:44,SCREEN,1.0,NaN,NaN,NaT,NaN,NaN,Mammography,SCREEN
115,4330344296,67156697.0,Heterogeneously dense (51% - 75%),NaN,4A - Suspicious abnormality - biopsy should be...,B-Biopsy should be considered,2021-01-25,2.0,NaN,NaN,NaN,NaN,NaN,470168.0,2021-02-10,Malignant,R,Mammography,DIAG
116,4330344296,67156697.0,Heterogeneously dense (51% - 75%),NaN,4A - Suspicious abnormality - biopsy should be...,B-Biopsy should be considered,2021-01-25,2.0,NaN,NaN,NaN,NaN,NaN,470167.0,2021-02-10,Benign,R,Mammography,DIAG
117,4330344296,67156697.0,Heterogeneously dense (51% - 75%),NaN,4A - Suspicious abnormality - biopsy should be...,B-Biopsy should be considered,2021-01-25,2.0,NaN,NaN,NaN,NaN,NaN,478775.0,2021-03-05,Malignant,R,Mammography,DIAG


In [51]:
# Create the final 'study' column by prioritizing cancer cohort (study info from DICOM), then patient_data_ie
merge['study'] = merge['study_x'].combine_first(merge['study_y'])
final = merge.drop(columns=['study_x', 'study_y'])

In [52]:
output_file = os.path.join("../Data/", study, 'cancer_cohort' + ".xlsx")
final.to_excel(output_file, index=False)

# II. Extract <span style="color:blue;">**MO cancer**</span> record

## <span style="color:#FF6347;">**READ**</span> file (cancer_cohort)

In [53]:
output_file = os.path.join("../Data/", "Cancer", 'cancer_cohort' + ".xlsx")
cancer_cohort = pd.read_excel(output_file)

In [54]:
PIDs = cancer_cohort["PATIENT_STUDY_ID"].unique()

In [55]:
len(PIDs)

1956

## 1. Locate <span style="color:blue;">**cancer**</span> index year, "index status" = <span style="color:#8A2BE2;">**INDEX**</span>

#### See [inclusion criteria][ic] to for filtering

[ic]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/doc.aspx?sourcedoc=%7B9c403413-a771-4830-98ab-15171872c4dd%7D&action=edit

#### Find the earliest "Malignant" record in pathology and marked the pathology date as <span style="color:#8A2BE2;">**INDEX**</span>, then drop all after

In [56]:
# Create a column called "index status"
cohort = cancer_cohort.copy()
cohort['index status'] = None

In [57]:
# 1. Sort to find the earliest Malignant PATHOLOGY_DATE
cohort_sorted_pathology = cohort.sort_values(
    ['PATIENT_STUDY_ID', 'PATHOLOGY_DATE']
)

# Filter for only 'Malignant' records by dropping duplicates
first_malignant_indices = cohort_sorted_pathology[
    cohort_sorted_pathology['LESION_CLASS'] == 'Malignant'
].drop_duplicates(
    subset=['PATIENT_STUDY_ID'],
    keep='first'
).index.tolist()

# 2. Mark the corresponding "index status" as "INDEX"
cohort.loc[first_malignant_indices, 'index status'] = 'INDEX'

# Create a clean target DataFrame for the next merge step
target_df = cohort.loc[first_malignant_indices].copy().rename(
    columns={'PATHOLOGY_DATE': 'target pathology date'}
)

#### Find the SCREEN/DIAGNOSIS exam that occurred within 1 year of the <span style="color:#8A2BE2;">**INDEX**</span> Pathology Date, marked as <span style="color:#8A2BE2;">**INDEX**</span>

In [58]:
# 3. Locate Exams within One Year of Index Pathology Date

tar_lookup = target_df.set_index('PATIENT_STUDY_ID')['target pathology date']
cohort['target pathology date'] = cohort['PATIENT_STUDY_ID'].map(tar_lookup)

# Calculate the time difference (Pathology Date - Exam Date)
# Negative difference means the exam occurred AFTER the pathology (we want to exclude these later)
cohort['path to exam diff'] = cohort['target pathology date'] - cohort['EXAM_COMPLETED_DATE']

# Create a boolean mask for exams within 1 year (365 days) BEFORE or ON the pathology date
one_year = pd.Timedelta(days=365)
# Condition 1: Exam occurred before or on the pathology date (Pathology - Exam >= 0)
mask_preceding = cohort['path to exam diff'] >= pd.Timedelta(days=0)
# Condition 2: Exam occurred within 1 year before the pathology date (Pathology - Exam <= 365 days)
mask_within_year = cohort['path to exam diff'] <= one_year
# Combined mask: Preceding AND within 1 year
mask_index_cohort = mask_preceding & mask_within_year

In [59]:
# Mark all rows matching the criteria as "index" (overwriting 'Other', retaining the original 'index' marks)
cohort.loc[mask_index_cohort, 'index status'] = 'INDEX'

print(f"✅ Step 3 Complete. Marked {cohort[cohort['index status'] == 'INDEX'].shape[0]} rows as 'INDEX'.")

✅ Step 3 Complete. Marked 7040 rows as 'INDEX'.


In [60]:
cohort['index status'].unique()

array(['INDEX', None], dtype=object)

In [61]:
cohort[cohort["PATIENT_STUDY_ID"]==4330344296]

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count,StudyDate,StudyTime,image_available,dbt_available,BX_ID,PATHOLOGY_DATE,LESION_CLASS,SIDE,EXAM_TYPE,study,index status,target pathology date,path to exam diff
108,4330344296,71644307.0,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2017-07-31,2.0,2017-07-31,02:23:29,1.0,NaN,NaN,NaT,NaN,NaN,Mammography,SCREEN,None,2021-02-10,1290 days
109,4330344296,76610889.0,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2018-10-31,1.0,2018-10-31,16:35:19,1.0,NaN,NaN,NaT,NaN,NaN,Mammography,SCREEN,None,2021-02-10,833 days
110,4330344296,76610889.0,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2018-10-31,1.0,2018-10-31,16:35:19,1.0,NaN,NaN,NaT,NaN,NaN,Mammography,SCREEN,None,2021-02-10,833 days
111,4330344296,76695188.0,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2018-11-08,1.0,2018-11-08,15:23:00,1.0,NaN,NaN,NaT,NaN,NaN,Mammography,DIAG,None,2021-02-10,825 days
112,4330344296,62861703.0,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2019-12-20,2.0,2019-12-20,02:44:43,1.0,NaN,NaN,NaT,NaN,NaN,Mammography,SCREEN,None,2021-02-10,418 days
113,4330344296,67469599.0,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2020-12-31,1.0,2020-12-31,08:39:44,1.0,NaN,NaN,NaT,NaN,NaN,Mammography,SCREEN,INDEX,2021-02-10,41 days
114,4330344296,67469599.0,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2020-12-31,1.0,2020-12-31,08:39:44,1.0,NaN,NaN,NaT,NaN,NaN,Mammography,SCREEN,INDEX,2021-02-10,41 days
115,4330344296,67156697.0,Heterogeneously dense (51% - 75%),NaN,4A - Suspicious abnormality - biopsy should be...,B-Biopsy should be considered,2021-01-25,2.0,NaN,NaN,NaN,NaN,470168.0,2021-02-10,Malignant,R,Mammography,DIAG,INDEX,2021-02-10,16 days
116,4330344296,67156697.0,Heterogeneously dense (51% - 75%),NaN,4A - Suspicious abnormality - biopsy should be...,B-Biopsy should be considered,2021-01-25,2.0,NaN,NaN,NaN,NaN,470167.0,2021-02-10,Benign,R,Mammography,DIAG,INDEX,2021-02-10,16 days
117,4330344296,67156697.0,Heterogeneously dense (51% - 75%),NaN,4A - Suspicious abnormality - biopsy should be...,B-Biopsy should be considered,2021-01-25,2.0,NaN,NaN,NaN,NaN,478775.0,2021-03-05,Malignant,R,Mammography,DIAG,INDEX,2021-02-10,16 days


## 2. Locate <span style="color:blue;">**MO cancer**</span> year, "index status" = <span style="color:#00BFFF;">**INDEX-1**</span>

#### See [inclusion criteria][ic] to for filtering

[ic]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/doc.aspx?sourcedoc=%7B9c403413-a771-4830-98ab-15171872c4dd%7D&action=edit

### 2a. Find the earliest <span style="color:#8A2BE2;">**INDEX**</span> exam date

#### Find the earliest EXAM_COMPLETED_DATE marked as <span style="color:#8A2BE2;">**"INDEX"**</span>

In [62]:
# 1. Isolate the current "INDEX" records
index_records = cohort[cohort['index status'] == 'INDEX'].copy()

# 2. Find the earliest EXAM_COMPLETED_DATE for the current "INDEX" cohort
earliest_index_exam = index_records.sort_values(
    ['PATIENT_STUDY_ID', 'EXAM_COMPLETED_DATE']
).drop_duplicates(
    subset=['PATIENT_STUDY_ID'],
    keep='first'
)

# Rename the earliest exam date for clarity in the merge
earliest_index_exam.rename(
    columns={'EXAM_COMPLETED_DATE': 'target index date'},
    inplace=True
)

In [63]:
earliest_index_exam

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,target index date,duplicate_count,StudyDate,StudyTime,image_available,dbt_available,BX_ID,PATHOLOGY_DATE,LESION_CLASS,SIDE,EXAM_TYPE,study,index status,target pathology date,path to exam diff
0,4330018595,63027507.0,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2019-07-26,2.0,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,Mammography,SCREEN,INDEX,2020-06-05,315 days
13,4330066079,61259016.0,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2020-01-14,2.0,2020-01-14,18:25:58,1.0,1.0,NaN,NaT,NaN,NaN,Mammography,DIAG,INDEX,2020-06-17,155 days
28,4330170072,79434020.0,Heterogeneously dense (51% - 75%),NaN,99 - Post procedure mammograms for marker plac...,X-Follow-up post biopsy as directed by clinician,2017-10-27,1.0,NaN,NaN,NaN,NaN,436484.0,2017-11-16,Malignant,L,Other,NaN,INDEX,2017-11-16,20 days
40,4330303540,73123621.0,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2016-12-21,2.0,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,Digital Mammography,SCREEN,INDEX,2017-01-27,37 days
48,4330325599,75132824.0,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2016-04-20,1.0,2016-04-20,02:32:42,1.0,1.0,NaN,NaT,NaN,NaN,Mammography,DIAG,INDEX,2016-04-20,0 days
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24795,4339508544,71405606.0,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2017-05-03,2.0,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,Digital Mammography,SCREEN,INDEX,2018-01-23,265 days
24816,4339577807,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,489989.0,2018-10-11,Malignant,L,NaN,NaN,INDEX,2018-10-11,NaT
24819,4339582412,78690585.0,Heterogeneously dense (51% - 75%),NaN,3 - Probably benign - short interval follow-up,F-Follow-up at short interval (1-11 months),2018-04-25,2.0,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,Other,NaN,INDEX,2018-05-08,13 days
24841,4339604670,68584905.0,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2020-11-27,1.0,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,Mammography,SCREEN,INDEX,2020-12-22,25 days


### 2b. Find the preceding exam, <span style="color:blue;">**MO cancer**</span> year, <span style="color:#00BFFF;">**INDEX-1**</span>

#### Find the preceding exam that occurred at least 9 months before the earliest <span style="color:#8A2BE2;">**"INDEX"**</span> exam date but not before 18 months

In [64]:
# 3. Locate closest preceding exam, time difference between closest preceding exam date and Index Exam Date should fall in the range 9 months ~ 1.5 year

tar_lookup = earliest_index_exam.set_index('PATIENT_STUDY_ID')['target index date']
cohort['target index date'] = cohort['PATIENT_STUDY_ID'].map(tar_lookup)

# Calculate the time difference (INDEX Exam Date - Exam Date)
# Negative difference means the exam occurred AFTER the index (we want to exclude these later)
cohort['index to exam diff'] = cohort['target index date'] - cohort['EXAM_COMPLETED_DATE']


In [65]:
# Create a boolean mask for exams more than 9 months and less than 18 months BEFORE the index date
nine_month = pd.Timedelta(days=274)
eighteen_month = pd.Timedelta(days=548)
# Condition 1: Exam occurred before the INDEX exam date (INDEX Exam - Exam > 0)
mask_preceding = cohort['index to exam diff'] > pd.Timedelta(days=0)
# Condition 2: Exam occurred within or on 18 months before INDEX exam date (INDEX Exam - Exam <= 18 months)
mask_within_18_month = cohort['index to exam diff'] <= eighteen_month
# Condition 3: Exam occurred more than or on 9 months before INDEX exam date (INDEX Exam - Exam >= 9 months)
mask_more_9_month = cohort['index to exam diff'] >= nine_month
# Combined mask: Preceding AND within 18 months AND more than 9 months
mask_index_1_cohort = mask_preceding & mask_within_18_month & mask_more_9_month

In [66]:
# Mark all rows matching the criteria as "index" (overwriting 'Other', retaining the original 'index' marks)
cohort.loc[mask_index_1_cohort, 'index status'] = 'INDEX-1'

print(f"✅ Step 4 Complete. Marked {cohort[cohort['index status'] == 'INDEX-1'].shape[0]} rows as 'INDEX-1'.")

✅ Step 4 Complete. Marked 1198 rows as 'INDEX-1'.


In [67]:
cohort["index status"].unique()

array(['INDEX', None, 'INDEX-1'], dtype=object)

In [68]:
cohort_sort = cohort.sort_values(['PATIENT_STUDY_ID', 'EXAM_COMPLETED_DATE', 'ACCESSION_NUMBER']).reset_index(drop=True)

In [69]:
cohort_sort[cohort_sort["PATIENT_STUDY_ID"]==4330344296]

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count,StudyDate,StudyTime,image_available,dbt_available,BX_ID,PATHOLOGY_DATE,LESION_CLASS,SIDE,EXAM_TYPE,study,index status,target pathology date,path to exam diff,target index date,index to exam diff
108,4330344296,71644307.0,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2017-07-31,2.0,2017-07-31,02:23:29,1.0,NaN,NaN,NaT,NaN,NaN,Mammography,SCREEN,None,2021-02-10,1290 days,2020-12-31,1249 days
109,4330344296,76610889.0,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2018-10-31,1.0,2018-10-31,16:35:19,1.0,NaN,NaN,NaT,NaN,NaN,Mammography,SCREEN,None,2021-02-10,833 days,2020-12-31,792 days
110,4330344296,76610889.0,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2018-10-31,1.0,2018-10-31,16:35:19,1.0,NaN,NaN,NaT,NaN,NaN,Mammography,SCREEN,None,2021-02-10,833 days,2020-12-31,792 days
111,4330344296,76695188.0,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2018-11-08,1.0,2018-11-08,15:23:00,1.0,NaN,NaN,NaT,NaN,NaN,Mammography,DIAG,None,2021-02-10,825 days,2020-12-31,784 days
112,4330344296,62861703.0,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2019-12-20,2.0,2019-12-20,02:44:43,1.0,NaN,NaN,NaT,NaN,NaN,Mammography,SCREEN,INDEX-1,2021-02-10,418 days,2020-12-31,377 days
113,4330344296,67469599.0,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2020-12-31,1.0,2020-12-31,08:39:44,1.0,NaN,NaN,NaT,NaN,NaN,Mammography,SCREEN,INDEX,2021-02-10,41 days,2020-12-31,0 days
114,4330344296,67469599.0,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2020-12-31,1.0,2020-12-31,08:39:44,1.0,NaN,NaN,NaT,NaN,NaN,Mammography,SCREEN,INDEX,2021-02-10,41 days,2020-12-31,0 days
115,4330344296,67156697.0,Heterogeneously dense (51% - 75%),NaN,4A - Suspicious abnormality - biopsy should be...,B-Biopsy should be considered,2021-01-25,2.0,NaN,NaN,NaN,NaN,470168.0,2021-02-10,Malignant,R,Mammography,DIAG,INDEX,2021-02-10,16 days,2020-12-31,-25 days
116,4330344296,67156697.0,Heterogeneously dense (51% - 75%),NaN,4A - Suspicious abnormality - biopsy should be...,B-Biopsy should be considered,2021-01-25,2.0,NaN,NaN,NaN,NaN,470167.0,2021-02-10,Benign,R,Mammography,DIAG,INDEX,2021-02-10,16 days,2020-12-31,-25 days
117,4330344296,67156697.0,Heterogeneously dense (51% - 75%),NaN,4A - Suspicious abnormality - biopsy should be...,B-Biopsy should be considered,2021-01-25,2.0,NaN,NaN,NaN,NaN,478775.0,2021-03-05,Malignant,R,Mammography,DIAG,INDEX,2021-02-10,16 days,2020-12-31,-25 days


In [70]:
output_file = os.path.join("../Data/", study, 'cancer_cohort_w_index' + ".xlsx")
cohort_sort.to_excel(output_file, index=False)

## 3. Extract <span style="color:blue;">**MO cancer**</span> cohort only

### <span style="color:#FF6347;">**READ**</span> file (cancer_cohort_w_index)

In [4]:
output_file = os.path.join("../Data/", study, 'cancer_cohort_w_index' + ".xlsx")
cohort = pd.read_excel(output_file)

In [5]:
PIDs = cohort["PATIENT_STUDY_ID"].unique()

### **Criteria**:
#### 1. At <span style="color:#8A2BE2;">**INDEX**</span>, "LESION_CLASS" = "Malignant"
##### no need to check as all patients in cancer_cohort already meet this criteria
#### 2. At <span style="color:#00BFFF;">**INDEX-1**</span>, "image_available" = 1
#### 3. At <span style="color:#00BFFF;">**INDEX-1**</span>, "FINDING_CATEGORY" = "1 - Negative", "2 - Benign finding"
#### 4. At <span style="color:#00BFFF;">**INDEX-1**</span>, "study" = "SCREEN"

In [6]:
idx_index_1 = cohort["index status"]=="INDEX-1"
idx_img = cohort["image_available"]==1
idx_dbt = cohort["dbt_available"]==1
idx_screen = cohort["study"]=="SCREEN"
idx_birads_0 = cohort["FINDING_CATEGORY"]=="0 - Need additional imaging evaluation"
idx_birads_1 = cohort["FINDING_CATEGORY"]=="1 - Negative"
idx_birads_2 = cohort["FINDING_CATEGORY"]=="2 - Benign finding"

#### [N=384] Meet **criteria #2**

In [7]:
pids_criteria_2 = cohort[idx_index_1 & idx_img]["PATIENT_STUDY_ID"].unique()

In [8]:
len(pids_criteria_2)

384

#### [N=378] Meet **criteria #2 <span style="color:red">**AND**</span> #3**

In [9]:
pids_criteria_2_3 = cohort[(idx_index_1 & idx_img & idx_birads_1) | (idx_index_1 & idx_img & idx_birads_2)]["PATIENT_STUDY_ID"].unique()

In [10]:
len(pids_criteria_2_3)

378

#### [N=31] Meet **criteria #2** <span style="color:red">**AND**</span> At <span style="color:#00BFFF;">**INDEX-1**</span>, "FINDING_CATEGORY" = "0 - Need additional imaging evaluation"

In [11]:
pids_criteria_2_birads_0 = cohort[idx_index_1 & idx_img & idx_birads_0]["PATIENT_STUDY_ID"].unique()

In [12]:
len(pids_criteria_2_birads_0)

31

In [13]:
len(set(pids_criteria_2_3.tolist()) & set(pids_criteria_2_birads_0.tolist()))

31

##### <span style="color:#FF6347;">**SAVE**</span>

In [14]:
cohort_filtered = cohort[cohort['PATIENT_STUDY_ID'].isin(pids_criteria_2_birads_0)].reset_index(drop=True)
cohort_filtered_sort = cohort_filtered.sort_values(['PATIENT_STUDY_ID', 'EXAM_COMPLETED_DATE', 'ACCESSION_NUMBER']).reset_index(drop=True)
output_file = os.path.join("../Data/", study, 'mo_cancer_cohort_N_31' + ".xlsx")
cohort_filtered.to_excel(output_file, index=False)

#### [N=219] Meet **criteria #2 <span style="color:red">**AND**</span> #3 <span style="color:red">**AND**</span> #4**  → Remove any BIRADS=0 at <span style="color:#00BFFF;">**INDEX-1**</span> → <span style="color:#FF6347;">**SAVE AS**</span> **mo_cancer_cohort**

##### Meet **criteria #2 <span style="color:red">**AND**</span> #3 <span style="color:red">**AND**</span> #4**

In [15]:
pids_criteria_all = cohort[(idx_screen & idx_index_1 & idx_img & idx_birads_1) | (idx_screen & idx_index_1 & idx_img & idx_birads_2)]["PATIENT_STUDY_ID"].unique()

In [16]:
len(pids_criteria_all)

251

##### Remove any BIRADS=0 at <span style="color:#00BFFF;">**INDEX-1**</span> 

In [17]:
pids_birads_0 = cohort[idx_index_1 & idx_birads_0]["PATIENT_STUDY_ID"].unique()

In [18]:
set_pids_common = set(pids_criteria_all) & set(pids_birads_0)

In [19]:
pids_mo = list(set(pids_criteria_all).difference(set_pids_common))

In [20]:
len(pids_mo)

219

#### [N=20] Meet **criteria #2 <span style="color:red">**AND**</span> #3 <span style="color:red">**AND**</span> #4** <span style="color:red">**AND**</span> **DBT available** → Remove any BIRADS=0 at <span style="color:#00BFFF;">**INDEX-1**</span>

##### Meet **criteria #2 <span style="color:red">**AND**</span> #3 <span style="color:red">**AND**</span> #4**<span style="color:red">**AND**</span> DBT available

In [21]:
pids_criteria_all_dbt = cohort[(idx_dbt & idx_screen & idx_index_1 & idx_img & idx_birads_1) | (idx_dbt & idx_screen & idx_index_1 & idx_img & idx_birads_2)]["PATIENT_STUDY_ID"].unique()

In [22]:
len(pids_criteria_all_dbt)

22

##### Remove any BIRADS=0 at <span style="color:#00BFFF;">**INDEX-1**</span> 

In [23]:
set_pids_common = set(pids_criteria_all_dbt) & set(pids_birads_0)

In [24]:
pids_mo_dbt = list(set(pids_criteria_all_dbt).difference(set_pids_common))

In [25]:
len(pids_mo_dbt)

20

##### <span style="color:#FF6347;">**SAVE**</span>

In [26]:
cohort_filtered = cohort[cohort['PATIENT_STUDY_ID'].isin(pids_mo_dbt)].reset_index(drop=True)
cohort_filtered_sort = cohort_filtered.sort_values(['PATIENT_STUDY_ID', 'EXAM_COMPLETED_DATE', 'ACCESSION_NUMBER']).reset_index(drop=True)
output_file = os.path.join("../Data/", study, 'mo_cancer_cohort_dbt' + ".xlsx")
cohort_filtered.to_excel(output_file, index=False)

#### <span style="color:#FF6347;">**SAVE**</span> **mo_cancer_cohort**

In [186]:
cohort_filtered = cohort[cohort['PATIENT_STUDY_ID'].isin(pids_mo)].reset_index(drop=True)

In [187]:
cohort_filtered[cohort_filtered["PATIENT_STUDY_ID"]==4337724790]

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count,StudyDate,StudyTime,image_available,dbt_available,BX_ID,PATHOLOGY_DATE,LESION_CLASS,SIDE,EXAM_TYPE,study,index status,target pathology date,path to exam diff,target index date,index to exam diff
2821,4337724790,63408900.0,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2019-06-01,2.0,2019-06-01,15:09:20,1.0,NaN,NaN,NaT,NaN,NaN,Mammography,SCREEN,NaN,2022-07-21,1146.0,2021-08-03,794.0
2822,4337724790,60961176.0,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2020-06-02,2.0,2020-06-02,09:37:45,1.0,NaN,NaN,NaT,NaN,NaN,Mammography,SCREEN,INDEX-1,2022-07-21,779.0,2021-08-03,427.0
2823,4337724790,455418188.0,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2021-06-19,1.0,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,Mammography,SCREEN,NaN,2022-07-21,397.0,2021-08-03,45.0
2824,4337724790,455418188.0,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2021-06-19,1.0,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,Mammography,SCREEN,NaN,2022-07-21,397.0,2021-08-03,45.0
2825,4337724790,455075546.0,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2021-08-03,1.0,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,Mammography,DIAG,INDEX,2022-07-21,352.0,2021-08-03,0.0
2826,4337724790,451418694.0,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2022-06-30,1.0,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,Mammography,SCREEN,INDEX,2022-07-21,21.0,2021-08-03,-331.0
2827,4337724790,451418694.0,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2022-06-30,1.0,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,Mammography,SCREEN,INDEX,2022-07-21,21.0,2021-08-03,-331.0
2828,4337724790,451048623.0,Heterogeneously dense (51% - 75%),NaN,4B - Suspicious abnormality - biopsy should be...,B-Biopsy should be considered,2022-07-13,1.0,2022-07-13,01:55:07,1.0,NaN,352533.0,2022-07-21,Malignant,L,Mammography,DIAG,INDEX,2022-07-21,8.0,2021-08-03,-344.0
2829,4337724790,450750121.0,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2022-10-25,1.0,NaN,NaN,NaN,NaN,351055.0,2022-10-25,Malignant,L,Interventional procedure,NaN,NaN,2022-07-21,-96.0,2021-08-03,-448.0


In [188]:
cohort_filtered_sort = cohort_filtered.sort_values(['PATIENT_STUDY_ID', 'EXAM_COMPLETED_DATE', 'ACCESSION_NUMBER']).reset_index(drop=True)

In [189]:
output_file = os.path.join("../Data/", study, 'mo_cancer_cohort' + ".xlsx")
cohort_filtered.to_excel(output_file, index=False)

## <span style="color:#FF6347;">**READ**</span> file

In [193]:
output_file = os.path.join("../Data/", study, 'mo_cancer_cohort' + ".xlsx")
cohort = pd.read_excel(output_file)

In [194]:
PIDs = cohort["PATIENT_STUDY_ID"].unique()

In [201]:
len(PIDs)

219